<a href="https://colab.research.google.com/github/kuds/rl-doom/blob/main/notebooks/01_environment_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Environment Exploration

This notebook walks through setting up the ViZDoom environment, inspecting
observations, testing wrappers (frame stacking, resizing), and running a
random-agent baseline.

## 1. Install & verify dependencies

In [ ]:
# --- Colab Setup ---
Uncomment the block below when running on Google Colab
import subprocess, os
if not os.path.exists("/content/rl-doom"):
    subprocess.run(["git", "clone", "https://github.com/kuds/rl-doom.git", "/content/rl-doom"], check=True)
os.chdir("/content/rl-doom/notebooks")
subprocess.run(["pip", "install", "-q", "-e", "/content/rl-doom[notebooks]"], check=True)

import vizdoom
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

print(f"ViZDoom version: {vizdoom.__version__}")

In [ ]:
# Google Drive integration for persistent storage on Colab
# Uncomment the block below when running on Google Colab
# ---
import shutil
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = "/content/drive/MyDrive/rl-doom"
os.makedirs(DRIVE_ROOT, exist_ok=True)
for subdir in ["checkpoints", "logs", "figures", "media"]:
    drive_dir = f"{DRIVE_ROOT}/{subdir}"
    local_dir = os.path.abspath(f"../{subdir}")
    os.makedirs(drive_dir, exist_ok=True)
    if os.path.islink(local_dir):
        os.remove(local_dir)          # refresh stale symlink
    if os.path.isdir(local_dir):
        # Migrate any existing local artifacts into Drive, then replace with symlink
        for f in os.listdir(local_dir):
            src = os.path.join(local_dir, f)
            dst = os.path.join(drive_dir, f)
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(local_dir)
    os.symlink(drive_dir, local_dir)
print(f"Google Drive mounted. Artifacts will persist at: {DRIVE_ROOT}")
# ---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))

from rl_doom.env import DoomEnv, FrameStack, ResizeObservation, SkipFrame

## 2. Create a basic environment

We start with the **Basic** scenario — a single room where the agent must
shoot a monster as quickly as possible.

In [ ]:
env = DoomEnv(scenario="basic")
obs, info = env.reset(seed=42)

print(f"Observation shape : {obs.shape}")
print(f"Observation dtype : {obs.dtype}")
print(f"Action space      : {env.action_space}")
print(f"Available actions : {env.available_actions}")

## 3. Visualize raw observations

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

obs, info = env.reset(seed=42)
axes[0].imshow(obs)
axes[0].set_title("After reset")

for i in range(1, 4):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    axes[i].set_title(f"Step {i} (a={action}, r={reward:.1f})")
    axes[i].imshow(obs)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/01_raw_observations.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Test wrappers

Apply **ResizeObservation** (84×84 grayscale), **SkipFrame** (repeat=4), and
**FrameStack** (4 frames) — the standard Atari-style preprocessing pipeline.

In [ ]:
env_wrapped = DoomEnv(scenario="basic")
env_wrapped = ResizeObservation(env_wrapped, shape=(84, 84))
env_wrapped = SkipFrame(env_wrapped, skip=4)
env_wrapped = FrameStack(env_wrapped, num_stack=4)

obs, info = env_wrapped.reset(seed=42)
print(f"Wrapped observation shape: {obs.shape}  (expect [4, 84, 84])")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for i in range(4):
    axes[i].imshow(obs[i], cmap="gray")
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")
plt.suptitle("Stacked grayscale frames after reset", y=1.02)
plt.tight_layout()
plt.savefig("../figures/01_stacked_frames.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Random-agent baseline

Run a random policy for multiple episodes and record total rewards. This gives
us a lower-bound to compare against trained agents.

In [ ]:
import time

N_EPISODES = 50
episode_rewards = []
episode_lengths = []

t_start = time.time()

for ep in range(N_EPISODES):
    obs, info = env_wrapped.reset(seed=ep)
    total_reward = 0.0
    steps = 0
    done = False
    while not done:
        action = env_wrapped.action_space.sample()
        obs, reward, terminated, truncated, info = env_wrapped.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated
    episode_rewards.append(total_reward)
    episode_lengths.append(steps)

wall_time = time.time() - t_start
total_steps = sum(episode_lengths)
fps = total_steps / wall_time if wall_time > 0 else 0

episode_rewards = np.array(episode_rewards)
episode_lengths = np.array(episode_lengths)

print(f"Random agent — {N_EPISODES} episodes")
print(f"  Mean reward : {episode_rewards.mean():.2f} ± {episode_rewards.std():.2f}")
print(f"  Min / Max   : {episode_rewards.min():.2f} / {episode_rewards.max():.2f}")
print(f"  Mean length : {episode_lengths.mean():.1f} ± {episode_lengths.std():.1f} steps")
print(f"  Wall time   : {wall_time:.1f}s | FPS: {fps:.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# -- Episode rewards --
axes[0].plot(episode_rewards, alpha=0.6, label="Episode reward")
axes[0].axhline(episode_rewards.mean(), color="red", linestyle="--", label="Mean")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Total Reward")
axes[0].set_title("Random Agent — Basic Scenario (Rewards)")
axes[0].legend()

# -- Episode lengths --
axes[1].plot(episode_lengths, alpha=0.6, color="green", label="Episode length")
axes[1].axhline(episode_lengths.mean(), color="red", linestyle="--", label="Mean")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Steps")
axes[1].set_title("Random Agent — Basic Scenario (Episode Length)")
axes[1].legend()

plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/01_random_baseline.png", dpi=150, bbox_inches="tight")
plt.show()

# Save baseline stats with reproducibility metadata
import platform, datetime

os.makedirs("../logs", exist_ok=True)
np.savez(
    "../logs/random_baseline_basic.npz",
    rewards=episode_rewards,
    episode_lengths=episode_lengths,
    mean=episode_rewards.mean(),
    std=episode_rewards.std(),
    # Reproducibility metadata
    seed_start=0,
    n_episodes=N_EPISODES,
    wall_time_seconds=wall_time,
    fps=fps,
    timestamp=str(datetime.datetime.now(datetime.timezone.utc)),
    python_version=platform.python_version(),
    platform=platform.platform(),
    torch_device="cpu",
)

## 6. Explore other scenarios

In [ ]:
scenarios = ["basic", "deadly_corridor", "defend_the_center", "deathmatch"]

for scenario in scenarios:
    try:
        e = DoomEnv(scenario=scenario)
        obs, _ = e.reset()
        print(f"{scenario:20s}  obs={obs.shape}  actions={e.action_space.n}")
        e.close()
    except Exception as ex:
        print(f"{scenario:20s}  ERROR: {ex}")

In [ ]:
env.close()
env_wrapped.close()
print("Done!")